In [1]:
from pathlib import Path
import yaml

repo_root = Path.cwd().resolve().parents[1]
print(repo_root)
config_path = repo_root / "configs" / "config.yaml"


with open(config_path, 'r') as config_file:
     config_file = yaml.safe_load(config_file)

data_dir = (repo_root / config_file["data_dir_relative_to_project_root"]).resolve()

/mnt/c/Users/samue/SynologyDrive/OceanPropInfSatImg


In [2]:
import copernicusmarine

username = config_file['copernicusmarine']['USERNAME']

copernicusmarine.subset(
  dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
  username=username,
  dataset_version="202311",
  variables=["bottomT", "mlotst", "siconc", "sithick", "so", "thetao", "uo", "usi", "vo", "vsi", "zos"],
  minimum_longitude=-156.07995709035455,
  maximum_longitude=-121.33383737365659,
  minimum_latitude=41.76973468648489,
  maximum_latitude=63.12744130133594,
  start_datetime="2021-06-25T00:00:00",
  end_datetime="2021-06-30T00:00:00",
  minimum_depth=0.49402499198913574,
  maximum_depth=0.49402499198913574,
  output_directory=data_dir,
  coordinates_selection_method="strict-inside",
  disable_progress_bar=True,
)

KeyboardInterrupt: 

In [2]:
from netCDF4 import Dataset

nc_ds = list(data_dir.glob('*'))
print(nc_ds[1])
glorysds_sample = Dataset(nc_ds[1])
for i, dim in enumerate(glorysds_sample.dimensions.keys()):
    print(f"Dimension #{i+1}: {dim}, Size: {glorysds_sample.dimensions[dim].size}")

for i, var in enumerate(glorysds_sample.variables.keys()):
    print(f"Variable #{i+1}: {var} {glorysds_sample.variables[var].long_name}, Size: {glorysds_sample.variables[var].size}, Units: {glorysds_sample.variables[var].units}, Shape: {glorysds_sample.variables[var].shape}")

/mnt/c/Users/samue/SynologyDrive/OceanPropInfSatImg/data/cmems_mod_glo_phy_my_0.083deg_P1D-m_multi-vars_156.00W-121.42W_41.83N-63.08N_0.49m_2021-06-25-2021-06-30.nc
Dimension #1: depth, Size: 1
Dimension #2: latitude, Size: 256
Dimension #3: longitude, Size: 416
Dimension #4: time, Size: 6
Variable #1: depth Depth, Size: 1, Units: m, Shape: (1,)
Variable #2: latitude Latitude, Size: 256, Units: degrees_north, Shape: (256,)
Variable #3: longitude Longitude, Size: 416, Units: degrees_east, Shape: (416,)
Variable #4: time Time, Size: 6, Units: hours since 1950-01-01, Shape: (6,)
Variable #5: bottomT Sea floor potential temperature, Size: 638976, Units: degrees_C, Shape: (6, 256, 416)
Variable #6: mlotst Density ocean mixed layer thickness, Size: 638976, Units: m, Shape: (6, 256, 416)
Variable #7: siconc Ice concentration, Size: 638976, Units: 1, Shape: (6, 256, 416)
Variable #8: sithick Sea ice thickness, Size: 638976, Units: m, Shape: (6, 256, 416)
Variable #9: so Salinity, Size: 638976,

In [3]:
import numpy as np
thetao = glorysds_sample.variables['thetao'][:]
thetao = np.squeeze(thetao, axis=1)
thetao.shape

(6, 256, 416)

In [4]:
mlds = glorysds_sample.variables['mlotst'][:]
mlds.shape

(6, 256, 416)

In [5]:
mlds_1_day = mlds[0]
mlds_1_day

masked_array(
  data=[[10.681478828191757, 10.681478828191757, 10.528886273503304, ...,
         --, --, --],
        [10.528886273503304, 10.528886273503304, 10.528886273503304, ...,
         --, --, --],
        [10.528886273503304, 10.528886273503304, 10.528886273503304, ...,
         --, --, --],
        ...,
        [--, --, --, ..., --, --, --],
        [--, --, --, ..., --, --, --],
        [--, --, --, ..., --, --, --]],
  mask=[[False, False, False, ...,  True,  True,  True],
        [False, False, False, ...,  True,  True,  True],
        [False, False, False, ...,  True,  True,  True],
        ...,
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],
  fill_value=np.int16(-32767))

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import numpy as np


slider = widgets.IntSlider(value=0, min=0, max=mlds.shape[0]-1, step=1, description='Day')

def plot_var(axis, fig, day, var, var_name):
    vmin = 0
    vmax = np.max(var)
    axis.set_xlabel("Longitude")
    axis.set_ylabel("Latitude")
    im = axis.imshow(var[day], origin='lower', aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
    axis.set_xticks([])
    axis.set_yticks([])
    fig.colorbar(im, ax=axis, label=var_name)

def plot_day(day):
    fig, axs = plt.subplots(figsize=(15, 5), ncols=2)
    plot_var(axs[1], fig, day, mlds, "Depth of Mixed Ocean Layer (m)")
    plot_var(axs[0], fig, day, thetao, "Sea Water Pressure at Sea Floor (dbar)")

widgets.interact(plot_day, day=slider)

interactive(children=(IntSlider(value=0, description='Day', max=5), Output()), _dom_classes=('widget-interact'…

<function __main__.plot_day(day)>